In [ ]:
from analyzer import * 
import seaborn as sns 

sns.set_theme(
    style="whitegrid",
    context="paper",
) 

vv = VQAResults()   
human_vqa = vv.human 
model_vqa = vv.model 

mm = MMStarResults()

model_mmstar = mm.model

In [ ]:
from utils.vqa import bin_number 

def filter_answer_type(df, answer_type, col):
    
    df = df[df['model'].isin(pretrained_models + ['Humans'])]
    df = df[df['answer_type'] == answer_type].copy()
    df['output'] = df[col] 

    if answer_type == 'yes/no': 
        df = yes_or_no(df)
    else: 
        df = df[df['question_type'] != 'what time']
        df['output'] = df[col].apply(lambda x: pp.postprocess_answer(x.lower()))
        df['number'] = df['output'].apply(extract_number_or_other) 
        df['number_bin'] = df['number'].apply(bin_number) 

    return df 

pp = PostProcessor() 
h = filter_answer_type(vv.human, 'number', 'answer_normalized')
m = filter_answer_type(vv.model, 'number', 'output') 
gt = filter_answer_type(vv.model, 'number', 'multiple_choice_answer')  

plot_df = pd.concat([
    gt[['number_bin']].assign(source='Ground Truth'),
    h[['number_bin']].assign(source='Humans'),
    m[['number_bin']].assign(source='Models')
])
counts = (
    plot_df
    .value_counts(['source', 'number_bin'])
    .reset_index(name='count')
)

counts['proportion'] = (
    counts
    .groupby('source')['count']
    .transform(lambda x: x / x.sum())
) 
stack_df = (
    counts
    .pivot(index='source', columns='number_bin', values='proportion')
    .fillna(0)
)
colors = {
    '0': '#DADADA',
    '1': '#BFD7EA',
    '2–3': '#9EC1A3',
    '4–5': '#7FB3A2',
    '6–10': '#5C8D89',
    '11–20': '#4C6A92',
    '>20': '#8E5A5A',
    'others': '#B5B5B5'
}

order = ['0','1','2–3','4–5','6–10','11–20','>20','others'] 

fig, ax = plt.subplots(figsize=(10,4))

stack_df[order].plot(
    kind='barh',
    stacked=True,
    ax=ax,
    color=[colors[c] for c in order],
    width=0.85
)

ax.set_xlim(0, 1)
# ax.set_xlabel('Proportion')
ax.set_ylabel('')
ax.grid(False)

for spine in ['top', 'right', 'left']:
    ax.spines[spine].set_visible(False)

# percentage labels
for i, source in enumerate(stack_df.index):
    cum = 0
    for cat in order:
        val = stack_df.loc[source, cat]
        if val > 0.04:
            ax.text(
                cum + val / 2,
                i,
                f'{val*100:.1f}%',
                ha='center',
                va='center',
                color='white',
                fontsize=12,
                fontweight='bold'
            )
        cum += val
ax.tick_params(axis='y', labelsize=15)
ax.legend(
    bbox_to_anchor=(0.5, 1.1),
    loc='upper center',
    ncol=8,
    frameon=False, 
    fontsize=12
)

# plt.title('Numeric Answer Distribution (Binned & Normalized)')
plt.tight_layout()
plt.show()

In [ ]:
colors = {
    'yes': '#6BA292',     # muted teal
    'no': '#C97A6A',      # muted coral
    'others': '#B5B5B5'   # neutral gray
}

h = filter_answer_type(vv.human, 'yes/no', 'answer_normalized')
m = filter_answer_type(vv.model, 'yes/no', 'output') 
gt = filter_answer_type(vv.model, 'yes/no', 'multiple_choice_answer')

counts = (
    plot_df
    .value_counts(['source', 'y/n'])
    .reset_index(name='count')
)

counts['proportion'] = (
    counts
    .groupby('source')['count']
    .transform(lambda x: x / x.sum())
)

stack_df = (
    counts
    .pivot(index='source', columns='y/n', values='proportion')
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(6.5, 2.8))

stack_df[['yes', 'no', 'others']].plot(
    kind='barh',
    stacked=True,
    ax=ax,
    color=[colors[c] for c in ['yes', 'no', 'others']],
    width=0.8   # ✅ thicker bars
)

# --- Aesthetics ---
ax.set_xlim(0, 1)
# ax.set_xlabel('Proportion', fontsize=11)
ax.set_ylabel('')
ax.set_yticklabels(stack_df.index, fontsize=11)
ax.grid(False)

# Remove spines for clean look
for spine in ['top', 'right', 'left']:
    ax.spines[spine].set_visible(False)

# --- Percentage labels inside bars ---
for i, source in enumerate(stack_df.index):
    cum = 0
    for cat in ['yes', 'no', 'others']:
        val = stack_df.loc[source, cat]
        if val > 0.04:  # avoid clutter on tiny segments
            ax.text(
                cum + val / 2,
                i,
                f'{val*100:.1f}%',
                ha='center',
                va='center',
                color='white',
                fontsize=10,
                fontweight='bold'
            )
        cum += val

# --- Legend outside, 2 columns ---
ax.legend(
    loc='upper right',
    bbox_to_anchor=(0.7, 1.2),
    ncol=3,
    frameon=False
)

# plt.title('Yes / No Answer Distribution (Normalized)', fontsize=12)
plt.tight_layout()
plt.show()